
# RQ2 — What Factors Influence the Relationship Between Agent Skills and PR Acceptance?

**Research Question.** Does the relationship between agent skill adoption and AI-authored pull request acceptance vary with the match between the pull request's task and the skill, the compatibility between the agent and the skill's location, and the origin of the skill (copied or written locally)?

**Goal.** Investigate whether the relationship between agent skill adoption and PR acceptance varies according to three characteristics:

1. **Task–Skill Match:** Whether the available skills are relevant to the task performed by the AI agent.
2. **Agent–Skill Compatibility:** Whether the coding agent supports the location where the skill is stored.
3. **Skill Provenance:** Whether the skill was created locally or reused from another source.

**Datasets.**

- GitSkills — skill metadata, repository information, and skill histories.
- AIDev v5 — AI-authored pull requests and merge outcomes.
- Agent–Skill Compatibility Reference — supported skill locations by coding agent.
- RQ1 Outputs — previously identified and matched repositories.

**Approach.**

We first examine the available data and determine which RQ2 characteristics can be measured reliably.

We then construct the relevant variables, analyze PR acceptance across different skill characteristics, perform statistical comparisons, and generate figures.

We distinguish directly measured characteristics from proxies and document any limitations.

## 0. Configuration

Load the required libraries, define the project paths, and verify the availability of the datasets.


In [1]:

from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data/raw"
INTERIM = ROOT / "data/interim"
PROCESSED = ROOT / "data/processed"
EXTERNAL = ROOT / "data/external"
FIG = ROOT / "figures"

SKILLS = RAW / "gitskills_artifacts_light.parquet"
PRS = RAW / "aidev_v5_all_pr.parquet"
MATCHES = PROCESSED / "rq1_matched_pairs.csv"
COMPATIBILITY = EXTERNAL / "agent_skill_locations.csv"

for folder in [PROCESSED, FIG]:
    folder.mkdir(parents=True, exist_ok=True)

files = [SKILLS, PRS, MATCHES, COMPATIBILITY]

for path in files:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")

con = duckdb.connect()

print("\nRQ2 configuration completed.")


gitskills_artifacts_light.parquet: FOUND
aidev_v5_all_pr.parquet: FOUND
rq1_matched_pairs.csv: FOUND
agent_skill_locations.csv: FOUND

RQ2 configuration completed.



## 1. Inspect Available Data

Before analyzing RQ2, we examine the available information in GitSkills, AIDev v5, and the agent–skill compatibility reference.

Our research question investigates three factors:

1. **Task–Skill Match:** Whether a skill is relevant to the task performed in a pull request.
2. **Agent–Skill Compatibility:** Whether an AI coding agent supports the location of a repository's skills.
3. **Skill Provenance:** Whether a skill was created locally or reused from another source.

We first identify which variables are directly available and which require additional data collection.

This prevents us from using unsupported assumptions or treating approximate indicators as verified measurements.


In [2]:

skill_columns = con.execute("""
DESCRIBE SELECT * FROM read_parquet(?)
""", [str(SKILLS)]).df()

pr_columns = con.execute("""
DESCRIBE SELECT * FROM read_parquet(?)
""", [str(PRS)]).df()

compatibility = pd.read_csv(COMPATIBILITY)
matches = pd.read_csv(MATCHES)

print("GitSkills columns:")
display(skill_columns[["column_name", "column_type"]])

print("\nAIDev v5 columns:")
display(pr_columns[["column_name", "column_type"]])

print("\nCompatibility reference:")
print("Rows:", len(compatibility))
print("Agents:", compatibility["agent"].nunique())
print("Locations:", compatibility["location"].nunique())
display(compatibility.head())

print("\nRQ1 matched repositories:")
print("Matched pairs:", len(matches))
print("Unique treated repositories:", matches["treated_repo"].nunique())

print("\nRQ2 data inspection completed.")


GitSkills columns:


,column_name,column_type
0,repo_full_name,VARCHAR
1,path,VARCHAR
2,filename,VARCHAR
3,location_class,VARCHAR
4,file_sha,VARCHAR
5,dedup_primary,BIGINT
6,history_fetched,BIGINT
7,first_commit_at,VARCHAR
8,last_commit_at,VARCHAR
9,commit_count,BIGINT



AIDev v5 columns:


,column_name,column_type
0,id,BIGINT
1,agent,VARCHAR
2,state,VARCHAR
3,created_at,VARCHAR
4,closed_at,VARCHAR
5,merged_at,VARCHAR
6,repo_id,DOUBLE
7,repo_url,VARCHAR



Compatibility reference:
Rows: 22
Agents: 6
Locations: 10


,agent,location,status,since,surface_note,source_url,source_type,checked_on
0,Claude_Code,.claude/skills,confirmed,2025-10-16,Project skills; also nested <subdir>/.claude/s...,https://code.claude.com/docs/en/skills,official_docs,2026-10-25
1,Claude_Code,.agents/skills,not_read,NaN,Not listed in current official docs.,https://code.claude.com/docs/en/skills,official_docs,2026-10-25
2,Claude_Code,.github/skills,not_read,NaN,Not listed in current official docs.,https://code.claude.com/docs/en/skills,official_docs,2026-10-25
3,Claude_Code,.cursor/skills,not_read,NaN,Not listed in current official docs.,https://code.claude.com/docs/en/skills,official_docs,2026-10-25
4,Claude_Code,.codex/skills,not_read,NaN,Not listed in current official docs.,https://code.claude.com/docs/en/skills,official_docs,2026-10-25



RQ1 matched repositories:
Matched pairs: 424
Unique treated repositories: 424

RQ2 data inspection completed.



## 2. Build the Skill Study Sample

This section identifies the agent skills associated with the 424 treated repositories from RQ1.

We retain files named exactly `SKILL.md` located in standard skill directories (`canonical` or `skills-dir`).

For each skill, we collect its repository, file path, location class, content hash, available commit date, and structural metadata.

This creates the foundation for examining agent–skill compatibility and skill provenance.

**Important limitation.** GitSkills represents a repository snapshot. The presence of a skill in this dataset does not prove that a coding agent actually loaded or used it when creating a pull request.


In [3]:

treated_repos = matches[
    ["pair_id", "treated_repo", "adoption_ts"]
].rename(columns={"treated_repo": "repo"}).copy()

treated_repos["repo"] = treated_repos["repo"].str.lower()
treated_repos["adoption_ts"] = pd.to_datetime(
    treated_repos["adoption_ts"]
)

con.register("rq2_treated_repos", treated_repos)

skills_sample = con.execute("""
SELECT
    t.pair_id,
    t.repo,
    t.adoption_ts,
    s.path,
    s.location_class,
    s.file_sha,
    TRY_CAST(
        REPLACE(s.first_commit_at, 'Z', '')
        AS TIMESTAMP
    ) AS skill_first_ts,
    s.first_commit_author_type,
    s.history_fetched,
    s.frontmatter_valid,
    s.has_scripts,
    s.has_references
FROM rq2_treated_repos t
JOIN read_parquet(?) s
    ON LOWER(s.repo_full_name) = t.repo
WHERE s.filename = 'SKILL.md'
  AND s.location_class IN ('canonical', 'skills-dir')
""", [str(SKILLS)]).df()

print("RQ1 treated repositories:", len(treated_repos))
print("Skill records:", len(skills_sample))
print("Repositories with skills:", skills_sample["repo"].nunique())
print("Dated skill records:", skills_sample["skill_first_ts"].notna().sum())
print("Unique skill content hashes:", skills_sample["file_sha"].nunique())

location_summary = skills_sample.groupby(
    "location_class"
).agg(
    skills=("path", "count"),
    repositories=("repo", "nunique"),
    dated_skills=("skill_first_ts", "count")
).reset_index()

print("\nSkills by location:")
display(location_summary)

print("\nExample skill records:")
display(skills_sample.head())


RQ1 treated repositories: 424
Skill records: 2111
Repositories with skills: 424
Dated skill records: 2111
Unique skill content hashes: 2111

Skills by location:


,location_class,skills,repositories,dated_skills
0,canonical,2099,414,2099
1,skills-dir,12,12,12



Example skill records:


,pair_id,repo,adoption_ts,path,location_class,file_sha,skill_first_ts,first_commit_author_type,history_fetched,frontmatter_valid,has_scripts,has_references
0,10,degrassiaaron/influencerai-monorepo,2025-10-18 09:54:47,.claude/skills/development/SKILL.md,canonical,ba26373e97b6cc1736ffd3014154a58a78306dc5,2025-10-18 09:54:47,User,1,1,0,0
1,340,mariodavid/jiracle,2026-01-12 08:33:09,.claude/skills/quality-analyzer/SKILL.md,canonical,b9c146cf5adc8f22a736e558ae1082be1c0d8914,2026-01-12 08:33:09,User,1,1,0,0
2,367,bobbyquantum/inkweld,2026-01-09 14:43:41,.github/skills/task-verification/SKILL.md,skills-dir,df89b36820e015a858e06dbbf37ff5330b1fe8fa,2026-01-09 14:43:41,User,1,0,0,0
3,21,anveio/conveaux,2025-12-16 20:27:07,.claude/skills/kpi-pr-throughput/SKILL.md,canonical,b40c21d9378dd698db1486de980aa8f80e13f227,2025-12-17 06:13:21,User,1,1,0,0
4,340,mariodavid/jiracle,2026-01-12 08:33:09,.claude/skills/code-review/SKILL.md,canonical,e297264352a966815c19837b06fb53d4b019f94f,2026-01-12 08:33:09,User,1,1,0,0



## 3. Link Skills to AI-Authored Pull Requests

This section connects the treated repositories' agent skills with their AI-authored pull requests.

We focus on PRs created during the 30 days following each repository's skill adoption date, consistent with RQ1.

For each PR, we identify skills whose recorded first-commit date is no later than the PR creation date.

We retain the coding agent, PR creation date, 30-day merge outcome, and number of available skills.

This temporal restriction avoids associating pull requests with skills introduced later.

**Limitation.** A skill's recorded availability does not establish that the coding agent actually loaded or used it.


In [4]:

con.register("rq2_skill_records", skills_sample)

rq2_post_prs = con.execute("""
WITH post_prs AS (
    SELECT
        t.pair_id,
        t.repo,
        t.adoption_ts,
        p.id AS pr_id,
        p.agent,
        TRY_CAST(REPLACE(p.created_at, 'Z', '') AS TIMESTAMP) AS created_ts,
        TRY_CAST(REPLACE(p.merged_at, 'Z', '') AS TIMESTAMP) AS merged_ts
    FROM read_parquet(?) p
    JOIN rq2_treated_repos t
        ON LOWER(REPLACE(
            p.repo_url,
            'https://api.github.com/repos/',
            ''
        )) = t.repo
),
filtered AS (
    SELECT *
    FROM post_prs
    WHERE created_ts >= adoption_ts
      AND created_ts < adoption_ts + INTERVAL 30 DAY
      AND created_ts <= TIMESTAMP '2026-03-31 23:59:59'
)
SELECT
    p.pair_id,
    p.repo,
    p.pr_id,
    p.agent,
    p.created_ts,
    CASE
        WHEN p.merged_ts >= p.created_ts
         AND p.merged_ts < p.created_ts + INTERVAL 30 DAY
        THEN 1
        ELSE 0
    END AS merged_30d,
    COUNT(s.path) AS available_skills
FROM filtered p
LEFT JOIN rq2_skill_records s
    ON p.repo = s.repo
   AND s.skill_first_ts <= p.created_ts
GROUP BY
    p.pair_id,
    p.repo,
    p.pr_id,
    p.agent,
    p.created_ts,
    p.merged_ts
""", [str(PRS)]).df()

print("Post-adoption PRs:", len(rq2_post_prs))
print("Unique PRs:", rq2_post_prs["pr_id"].nunique())
print("Repositories:", rq2_post_prs["repo"].nunique())

print("PRs with at least one available skill:",
      (rq2_post_prs["available_skills"] >= 1).sum())

print("PRs without an available skill:",
      (rq2_post_prs["available_skills"] == 0).sum())

print("\nSkills available per PR:")
display(rq2_post_prs["available_skills"].describe().round(2))

print("\nPRs by coding agent:")
display(
    rq2_post_prs.groupby("agent").agg(
        prs=("pr_id", "count"),
        repositories=("repo", "nunique")
    ).sort_values("prs", ascending=False)
)

print("\nExample PR records:")
display(rq2_post_prs.head())


Post-adoption PRs: 7371
Unique PRs: 7371
Repositories: 317
PRs with at least one available skill: 7371
PRs without an available skill: 0

Skills available per PR:


count    7371.00
mean        3.71
std         4.12
min         1.00
25%         1.00
50%         2.00
75%         5.00
max        27.00
Name: available_skills, dtype: float64


PRs by coding agent:


,prs,repositories
agent,,
Claude_Code,6799,281
Copilot,346,58
OpenAI_Codex,117,28
Cursor,66,8
Google_Jules,43,6



Example PR records:


,pair_id,repo,pr_id,agent,created_ts,merged_30d,available_skills
0,111,yusuke-suzuki/dotfiles,3990515215,Claude_Code,2026-02-25 16:41:23,1,5
1,87,handley-lab/mcp-handley-lab,3957425534,Claude_Code,2026-02-18 11:41:38,1,1
2,91,m96-chan/pygpukit,3775068125,Claude_Code,2026-01-01 19:16:02,1,9
3,35,hyperlane-xyz/hyperlane-monorepo,3908495822,Claude_Code,2026-02-06 21:33:16,0,13
4,73,schmitthub/clawker,3912588125,Claude_Code,2026-02-08 11:44:59,1,1



## 4. Measure Agent–Skill Compatibility

This section examines whether the coding agent responsible for a pull request supports the location of the skills available in its repository.

We compare each skill path with the agent–skill compatibility reference.

Three categories are defined:

1. **Documented by PR date:** At least one available skill is stored in a location officially supported by the agent on or before the PR creation date.
2. **Confirmed, start date unknown:** Support is documented, but its historical availability at PR creation cannot be verified.
3. **Not verified:** The available evidence does not establish compatibility at PR creation.

We distinguish documented compatibility from actual skill usage. A compatible location does not prove that the agent loaded or followed a skill.

This classification will support an exploratory comparison of PR acceptance across compatibility categories.


In [5]:

con.register("rq2_post_prs_df", rq2_post_prs)

skill_links = con.execute("""
SELECT DISTINCT
    p.pr_id,
    p.agent,
    p.created_ts,
    s.path
FROM rq2_post_prs_df p
JOIN rq2_skill_records s
    ON p.repo = s.repo
   AND s.skill_first_ts <= p.created_ts
""").df()

skill_links["path_norm"] = (
    skill_links["path"]
    .str.replace("\\", "/", regex=False)
    .str.replace(r"^\./", "", regex=True)
    .str.lower()
)

skill_links["documented_by_pr_date"] = False
skill_links["confirmed_unknown_start"] = False

compatibility["since_ts"] = pd.to_datetime(
    compatibility["since"], errors="coerce"
)

for ref in compatibility.itertuples(index=False):
    if ref.status != "confirmed":
        continue

    location = str(ref.location).lower()

    agent_match = skill_links["agent"].eq(ref.agent)

    path_match = skill_links["path_norm"].str.startswith(
        location + "/"
    )

    if ref.agent == "Claude_Code" and location == ".claude/skills":
        path_match = path_match | skill_links["path_norm"].str.contains(
            "/.claude/skills/", regex=False
        )

    matching = agent_match & path_match

    if pd.notna(ref.since_ts):
        matching = matching & (
            skill_links["created_ts"] >= ref.since_ts
        )
        skill_links.loc[
            matching, "documented_by_pr_date"
        ] = True
    else:
        skill_links.loc[
            matching, "confirmed_unknown_start"
        ] = True

compat_counts = skill_links.groupby(
    "pr_id", as_index=False
).agg(
    documented_skills=("documented_by_pr_date", "sum"),
    uncertain_date_skills=("confirmed_unknown_start", "sum")
)

rq2_compatibility = rq2_post_prs.merge(
    compat_counts,
    on="pr_id",
    how="left",
    validate="one_to_one"
)

rq2_compatibility[
    ["documented_skills", "uncertain_date_skills"]
] = rq2_compatibility[
    ["documented_skills", "uncertain_date_skills"]
].fillna(0).astype(int)

rq2_compatibility["compatibility_category"] = np.select(
    [
        rq2_compatibility["documented_skills"] > 0,
        rq2_compatibility["uncertain_date_skills"] > 0
    ],
    [
        "Documented by PR date",
        "Confirmed, start date unknown"
    ],
    default="Not verified"
)

print("Total PRs:", len(rq2_compatibility))

print("\nCompatibility categories:")
display(
    rq2_compatibility["compatibility_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="prs")
)

print("\nCompatibility by coding agent:")
display(
    rq2_compatibility.groupby(
        ["agent", "compatibility_category"]
    ).agg(
        prs=("pr_id", "count"),
        repositories=("repo", "nunique")
    ).reset_index().sort_values(
        ["agent", "prs"], ascending=[True, False]
    )
)

assert len(rq2_compatibility) == len(rq2_post_prs)
assert rq2_compatibility["pr_id"].is_unique

print("\nRQ2 compatibility classification completed.")


Total PRs: 7371

Compatibility categories:


,category,prs
0,Documented by PR date,7043
1,Not verified,262
2,"Confirmed, start date unknown",66



Compatibility by coding agent:


,agent,compatibility_category,prs,repositories
0,Claude_Code,Documented by PR date,6787,278
1,Claude_Code,Not verified,12,3
2,Copilot,Documented by PR date,256,45
3,Copilot,Not verified,90,17
4,Cursor,"Confirmed, start date unknown",66,8
5,Google_Jules,Not verified,43,6
6,OpenAI_Codex,Not verified,117,28



RQ2 compatibility classification completed.
